# Assignment 2 · Diagnose and improve generated images
**VAE track · Notebook v8 · Chonnam National University · Fall 2026**

Choose **VAE OR GAN**, not both. / VAE 또는 GAN 중 **하나만** 선택하세요. Released 16 October 2026 · Due 4 November 2026, 23:59 KST · 12 points · 10% of course grade.

Inspect generated MNIST images, make one planned change, and judge its result. Both tracks train twice from the same initial seed, use a fixed `z` panel for comparison, and ask for three short pieces of training or generation code. Improved images are **not required** for full credit. / 생성 이미지의 문제를 찾고 변경 계획을 먼저 쓴 뒤 실험하세요. 결과가 개선되지 않아도 근거 있는 분석이면 만점을 받을 수 있습니다.

VAE: three runs compare KL weight β = 20, 1.0 and 0.1. Prior generation and input reconstruction answer different questions.

Active work target: **3–5 hours** including a 16-image audit per run and 1–2-page analysis. Three prescribed CPU training runs; no hyperparameter search. Submit the executed notebook and **one PDF** with your 1–2-page analysis followed by the two-page receipt. See `A2_Student_Assignment_v9.pdf`. English or Korean answers are accepted; technical terms can remain in English.

## 0 Setup and data / 준비
Run these supplied cells. `auto` downloads MNIST if needed. If download fails, an offline sklearn digits dataset runs as a smoke test, but its result is **not submission-ready**. No Drive mount, GPU, or paid service is required. Reference code is openly provided through `check_code(N, use_reference=True)` and has **no penalty**; explain each block in your report.

In [ ]:
#@title Setup and workflow / 준비 및 실행 관리 {display-mode:"form"}
import copy, gzip, hashlib, json, math, os, platform, struct, time, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, FileLink

torch.set_num_threads(2)
VERSION = 'A2 redesign v9 / notebook v8'
SEED = 0
BASE, VARIANT, SECOND, PLAN, AUDIT = None, None, None, None, None
CODE_STATUS = {}
OUT = Path('a2_outputs'); OUT.mkdir(exist_ok=True)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()[:16]

def array_id(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

def code_id():
    # Detect ordinary edits after a run; this is not an anti-tampering system.
    return fingerprint({name: (globals()[name].__code__.co_code.hex(), repr(globals()[name].__code__.co_consts)) for name in CODE_NAMES})

def check_code(number, use_reference=False):
    name = CODE_NAMES[number - 1]
    if use_reference:
        globals()[name] = REFERENCES[number - 1]
    try:
        test_student_code(number)
        mode = 'reference' if globals()[name] is REFERENCES[number - 1] else 'student'
        CODE_STATUS[name] = mode
        print(f'Code {number}: PASS ({mode})')
        return True
    except Exception as exc:
        CODE_STATUS[name] = 'pending'
        print(f'Code {number}: NOT READY. {type(exc).__name__}: {exc}')
        print('update, tensor shape, eval/no_grad를 확인하세요. 필요하면 check_code(number, use_reference=True)를 사용하세요. 감점은 없습니다.')
        return False

def record_plan(student_id, diagnosis, candidate, prediction, success_rule, second_prediction=None):
    global PLAN, VARIANT, SECOND, AUDIT
    if BASE is None:
        print('먼저 baseline을 실행하세요 / Run the baseline first.'); return
    values = [student_id, diagnosis, prediction, success_rule]
    if any(not isinstance(x, str) or not x.strip() for x in values):
        print('NOT READY: 학번, baseline 관찰, 예측, 판단 기준을 작성하세요.'); return
    if not isinstance(second_prediction,str) or not second_prediction.strip():
        print('NOT READY: 두 번째 변경에 대한 예측도 작성하세요.'); return
    if candidate not in OPTIONS:
        print('NOT READY: 이 track에서 허용된 변경값을 사용하세요 / use the assigned option.'); return
    draft = dict(student_id=student_id.strip(), diagnosis=diagnosis.strip(), candidate=float(candidate),
                 prediction=prediction.strip(), success_rule=success_rule.strip(), baseline_id=BASE['id'])
    draft['second_prediction']=second_prediction.strip()
    if PLAN != draft:
        PLAN = draft; VARIANT = SECOND = AUDIT = None
    print('PLAN SAVED. 변경 실험을 실행하세요 / Run your chosen intervention.')

def run_baseline():
    global BASE, VARIANT, SECOND, PLAN, AUDIT
    cfg = config(BASE_VALUE)
    if BASE is not None and BASE['config'] == cfg:
        print('기존 baseline 재사용 / Reusing baseline.'); return
    BASE = train(BASE_VALUE, student=False)
    PLAN = VARIANT = SECOND = AUDIT = None

def run_variant():
    global VARIANT, SECOND, AUDIT
    if PLAN is None or BASE is None:
        print('NOT READY: baseline을 보고 변경 계획을 먼저 작성하세요.'); return
    if BASE['config'] != config(BASE_VALUE) or PLAN['baseline_id'] != BASE['id']:
        print('NOT READY: 설정이 바뀌었습니다. baseline부터 다시 실행하세요.'); return
    if not all([check_code(i) for i in [1, 2, 3]]):
        print('NOT READY: 코드 세 칸을 완성하거나 제공 코드로 전환하세요.'); return
    key = fingerprint(PLAN)
    if VARIANT is not None and SECOND is not None and VARIANT['plan_id'] == key and VARIANT['code_id'] == code_id():
        print('기존 변경 실험 재사용 / Reusing intervention run.'); return
    VARIANT = train(PLAN['candidate'], student=True)
    VARIANT.update(plan_id=key, code_id=code_id())
    SECOND = train(PLAN['candidate'], student=True, g_updates=G_UPDATES_SECOND) if TRACK=='GAN' else train(SECOND_VALUE,student=True)
    SECOND.update(plan_id=key, code_id=code_id())
    AUDIT = None

def ready_reason():
    if BASE is None or PLAN is None or VARIANT is None:
        return 'baseline, plan 또는 intervention 실행이 남아 있습니다.'
    if BASE['config'] != config(BASE_VALUE) or VARIANT['config'] != config(PLAN['candidate']):
        return '설정이 바뀌었습니다. baseline부터 다시 실행하세요.'
    if VARIANT['plan_id'] != fingerprint(PLAN) or PLAN['baseline_id'] != BASE['id']:
        return '변경 계획과 실행 결과가 다릅니다.'
    if VARIANT['code_id'] != code_id() or any(CODE_STATUS.get(n) == 'pending' for n in CODE_NAMES):
        return '코드가 바뀌었습니다. 코드 점검과 변경 실험을 다시 실행하세요.'
    expected_second=config(PLAN['candidate'],G_UPDATES_SECOND) if TRACK=='GAN' else config(SECOND_VALUE)
    if (SECOND is None or SECOND['config'] != expected_second
        or SECOND['plan_id'] != fingerprint(PLAN) or SECOND['code_id'] != code_id()):
        return '두 번째 변경 실험이 남아 있거나 설정이 바뀌었습니다.'
    return ''

def make_receipt(ai_use):
    reason = ready_reason()
    if not reason and 'OFFLINE' in DATA_NAME:
        reason = '오프라인 smoke test입니다. 제출 전 MNIST로 두 run을 실행하세요.'
    if not reason and (not isinstance(ai_use, str) or not ai_use.strip()):
        reason = 'AI 사용 내용을 쓰세요. 사용하지 않았다면 none을 적으세요.'
    if not reason and (AUDIT is None or AUDIT['run_id'] != VARIANT['id'] or AUDIT.get('second_id') != SECOND['id']):
        reason = '생성 grid의 sample 01–16에 대한 visual audit를 작성하세요.'
    if reason:
        print('NOT READY:', reason); return None
    b, v = metrics(BASE), metrics(VARIANT)
    payload = dict(version=VERSION, track=TRACK, plan=PLAN, code=CODE_STATUS.copy(),
                   baseline=dict(config=BASE['config'], metrics=b, sample_id=array_id(BASE['samples'])),
                   intervention=dict(config=VARIANT['config'], metrics=v, sample_id=array_id(VARIANT['samples'])),
                   visual_audit=AUDIT, ai_use=ai_use.strip(),
                   environment=dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, device='cpu'))
    payload['second_change']=dict(config=SECOND['config'],metrics=metrics(SECOND),sample_id=array_id(SECOND['samples']))
    fid=fingerprint(payload)
    receipt=dict(status='READY', fingerprint=fid, **payload)
    (OUT/f'A2_{TRACK}_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf8')
    # English labels and numbers in the PDF avoid missing Korean fonts on Colab.
    # The full original Korean plan/AI statement is preserved in the notebook and JSON.
    lines=[f'{VERSION} | {TRACK} | READY',f'Fingerprint: {fid}',
           f'Data: {BASE["config"]["data"]}', f'Changed parameter: {PARAMETER}',
           f'Baseline: {BASE_VALUE}   First change: {PLAN["candidate"]}   Second change: {SECOND["config"]}',
           'All runs restart from the same seed; comparisons change one factor at a time.',
           'Settings: '+json.dumps(BASE['config'],ensure_ascii=True), '',
           'BASELINE', *[f'  {k}: {v}' for k,v in b.items()], '',
           'INTERVENTION', *[f'  {k}: {v}' for k,v in v.items()], '',
           'SECOND CHANGE',*[f'  {k}: {v}' for k,v in metrics(SECOND).items()],'',
           'Code: '+str(CODE_STATUS),
           'Full plan, AI statement and annotations: notebook and receipt JSON.',
           'This receipt is an appendix. Write your own analysis in the report.',
           'Fingerprint checks consistency; it does not prove authorship.']
    import textwrap
    wrapped='\n'.join(y for x in lines for y in (textwrap.wrap(x,95) or ['']))
    pdf_path=OUT/f'A2_{TRACK}_receipt.pdf'
    with PdfPages(pdf_path) as pdf:
        fig=plt.figure(figsize=(8.27,11.69));fig.text(.07,.95,wrapped,va='top',fontfamily='monospace',fontsize=8.5);pdf.savefig(fig);plt.close(fig)
        fig=comparison_figure();pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    print('READY | fingerprint:',fid)
    print('Baseline:',b); print('Intervention:',v)
    print('분석 보고서 뒤에 PDF를 붙이세요. JSON은 노트북과 함께 보관하면 됩니다.')
    display(FileLink(str(pdf_path)));display(FileLink(str(OUT/f'A2_{TRACK}_receipt.json')))
    return receipt

In [ ]:
#@title Model, data, training and figures / model, data, 학습 및 그림 {display-mode:"form"}
TRACK='VAE'
PARAMETER='KL weight beta (three levels)'
BASE_VALUE=20.0
OPTIONS=(1.0,)
SECOND_VALUE=0.1
EPOCHS=6; BATCH=256; LATENT=2; LR=0.001
DATA_SOURCE='auto'  # auto: MNIST; offline: bundled sklearn digits, smoke test only
DATA_NAME=None
Z_FIXED=torch.randn(64,LATENT,generator=torch.Generator().manual_seed(2026))

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(784,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU())
        self.head_mu=nn.Linear(256,LATENT);self.head_logvar=nn.Linear(256,LATENT)
        self.dec=nn.Sequential(nn.Linear(LATENT,256),nn.ReLU(),nn.Linear(256,512),nn.ReLU(),nn.Linear(512,784))
    def encode(self,x):
        h=self.enc(x);return self.head_mu(h),self.head_logvar(h)
    def decode(self,z):return self.dec(z)

def kl_term(mu,lv):
    return (.5*(mu.square()+lv.exp()-1-lv)).sum(1).mean()

def reference_vae_update(model,opt,x,beta,eps):
    mu,lv=model.encode(x)
    z=mu+torch.exp(.5*lv)*eps
    rec=F.binary_cross_entropy_with_logits(model.decode(z),x,reduction='sum')/len(x)
    kl=kl_term(mu,lv)
    opt.zero_grad();(rec+beta*kl).backward();opt.step()
    return float(rec.detach()),float(kl.detach())

def reference_generate_prior(model,z):
    was_training=model.training;model.eval()
    with torch.no_grad(): out=torch.sigmoid(model.decode(z)).cpu().numpy()
    model.train(was_training)
    return out

def reference_reconstruct_mean(model,x):
    was_training=model.training;model.eval()
    with torch.no_grad():
        mu,_=model.encode(x)
        out=torch.sigmoid(model.decode(mu)).cpu().numpy()
    model.train(was_training)
    return out

CODE_NAMES=['vae_update','generate_prior','reconstruct_mean']
REFERENCES=[reference_vae_update,reference_generate_prior,reference_reconstruct_mean]

def test_student_code(number):
    fn=globals()[CODE_NAMES[number-1]]
    if number==1:
        torch.manual_seed(791);m1=VAE();m2=copy.deepcopy(m1)
        o1=torch.optim.SGD(m1.parameters(),lr=.005);o2=torch.optim.SGD(m2.parameters(),lr=.005)
        x=torch.rand(4,784);eps=torch.randn(4,LATENT)
        expected=reference_vae_update(m1,o1,x,1.,eps)
        actual=fn(m2,o2,x,1.,eps)
        assert actual is not None and len(actual)==2, 'return reconstruction BCE and KL'
        assert np.allclose(actual,expected,rtol=2e-4,atol=2e-4), 'BCE/KL computation differs'
        assert all(torch.allclose(a,b,rtol=2e-4,atol=2e-5) for a,b in zip(m1.parameters(),m2.parameters())), 'optimizer update differs'
    else:
        torch.manual_seed(792);m=VAE();m.train();z=torch.randn(5,LATENT);x=torch.rand(5,784)
        before=[p.clone() for p in m.parameters()]
        expected=(reference_generate_prior if number==2 else reference_reconstruct_mean)(m,z if number==2 else x)
        actual=fn(m,z if number==2 else x)
        assert actual is not None and isinstance(actual,np.ndarray), 'return a NumPy image array'
        assert actual.shape==expected.shape and np.allclose(actual,expected,rtol=1e-5,atol=1e-6), 'generated images differ'
        assert m.training and all(torch.equal(a,b) for a,b in zip(before,m.parameters())), 'restore model mode and do not update weights'

def load_data(source='auto'):
    global X_TRAIN,X_TEST,DATA_NAME,DATA_SOURCE
    DATA_SOURCE=source
    if source not in ['auto','offline']:raise ValueError('source must be auto or offline')
    try:
        if source=='offline':raise OSError('offline mode requested')
        arrays=[]
        for name in ['train-images-idx3-ubyte.gz','t10k-images-idx3-ubyte.gz']:
            p=Path(name)
            if not p.exists():
                last=None
                for base in ['https://ossci-datasets.s3.amazonaws.com/mnist/','https://storage.googleapis.com/cvdf-datasets/mnist/']:
                    try:
                        with urllib.request.urlopen(base+name,timeout=8) as response:raw=response.read()
                        # Validate before caching; a failed download must not become a valid cache.
                        decoded=gzip.decompress(raw);magic,n,r,c=struct.unpack('>IIII',decoded[:16])
                        assert magic==2051 and r==28 and c==28 and len(decoded)==16+n*r*c
                        p.write_bytes(raw);break
                    except Exception as exc:last=exc
                else:raise OSError(f'MNIST download failed: {last}')
            with gzip.open(p,'rb') as f:
                magic,n,r,c=struct.unpack('>IIII',f.read(16));assert magic==2051 and r*c==784
                arrays.append(torch.from_numpy(np.frombuffer(f.read(),np.uint8).copy().reshape(n,784)).float()/255)
        X_TRAIN,X_TEST=arrays;DATA_NAME='MNIST 60000 train / 10000 test'
    except (OSError,AssertionError,ValueError,EOFError,struct.error) as exc:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        digits=load_digits();x=torch.tensor(digits.images/16,dtype=torch.float32)[:,None]
        x=F.interpolate(x,size=(28,28),mode='bilinear',align_corners=False).flatten(1)
        tr,te=train_test_split(np.arange(len(x)),test_size=.2,random_state=42,stratify=digits.target)
        X_TRAIN,X_TEST=x[tr],x[te];DATA_NAME='OFFLINE sklearn digits SMOKE TEST ONLY'
        print('MNIST unavailable:',type(exc).__name__)
        print('오프라인 실행 점검용 data입니다. MNIST 결과로 해석하거나 제출하지 마세요.')
    print(DATA_NAME,tuple(X_TRAIN.shape),tuple(X_TEST.shape))

def config(value):
    return dict(data=DATA_NAME,seed=SEED,epochs=EPOCHS,batch=BATCH,latent=LATENT,lr=LR,beta=float(value),
                train_size=len(X_TRAIN),test_size=len(X_TEST),optimizer='Adam default betas',eval_noise_seed=2026)

def train(value,student=False):
    start=time.time();torch.manual_seed(SEED);model=VAE();opt=torch.optim.Adam(model.parameters(),lr=LR)
    history=[]
    for ep in range(1,EPOCHS+1):
        perm=torch.randperm(len(X_TRAIN));rt=kt=0.
        for i in range(0,len(X_TRAIN),BATCH):
            x=X_TRAIN[perm[i:i+BATCH]];eps=torch.randn(len(x),LATENT)
            rec,kl=(vae_update if student else reference_vae_update)(model,opt,x,value,eps)
            rt+=rec*len(x);kt+=kl*len(x)
        history.append([ep,rt/len(X_TRAIN),kt/len(X_TRAIN)])
        print(f'epoch {ep}/{EPOCHS} | reconstruction BCE {history[-1][1]:.2f} | KL {history[-1][2]:.2f}')
    model.eval()
    with torch.no_grad():
        mu,lv=model.encode(X_TEST)
        eps=torch.randn(mu.shape,generator=torch.Generator().manual_seed(12345))
        z=mu+torch.exp(.5*lv)*eps
        rec=float(F.binary_cross_entropy_with_logits(model.decode(z),X_TEST,reduction='sum'))/len(X_TEST)
        kl=float((.5*(mu.square()+lv.exp()-1-lv)).sum(1).mean())
    samples=(generate_prior if student else reference_generate_prior)(model,Z_FIXED)
    recon=(reconstruct_mean if student else reference_reconstruct_mean)(model,X_TEST[:8])
    assert np.isfinite(samples).all()
    result=dict(config=config(value),samples=samples,recon=recon,history=history,rec=rec,kl=kl,seconds=round(time.time()-start,2))
    result['id']=fingerprint([result['config'],array_id(samples)])
    print('Completed in',result['seconds'],'s. beta =',value)
    return result

def image_grid(fig,slot,images,title,number=True,rows=4):
    sub=slot.subgridspec(rows,8,wspace=.03,hspace=.3)
    for i in range(min(rows*8,len(images))):
        ax=fig.add_subplot(sub[i//8,i%8]);ax.imshow(images[i].reshape(28,28),cmap='gray',vmin=0,vmax=1);ax.axis('off')
        if number:ax.set_title(f'{i+1:02d}',fontsize=7,pad=1)
    # Group title is drawn by the caller, outside the image axes.

def baseline_figure():
    if BASE is None:return
    fig=plt.figure(figsize=(10,6));gs=fig.add_gridspec(1,2)
    image_grid(fig,gs[0],X_TEST[:32].numpy(),'Real test images');image_grid(fig,gs[1],BASE['samples'][:32],'Baseline')
    fig.text(.26,.97,'Real test images',ha='center',fontsize=12);fig.text(.75,.97,'Baseline | prior generations',ha='center',fontsize=12)
    plt.show();plt.close(fig)

def comparison_figure():
    fig=plt.figure(figsize=(15,9));gs=fig.add_gridspec(2,3,height_ratios=[4,3],hspace=.40,wspace=.10)
    for col,(run,title) in enumerate([(BASE,'A · beta 20'),(VARIANT,'B · beta 1'),(SECOND,'C · beta 0.1')]):
        image_grid(fig,gs[0,col],run['samples'][:32],title)
        fig.text((col+.5)/3,.95,title+' · prior from same z',ha='center',fontsize=11)
    lower=gs[1,:].subgridspec(4,8,wspace=.03,hspace=.15)
    for row,images in enumerate([X_TEST[:8].numpy(),BASE['recon'],VARIANT['recon'],SECOND['recon']]):
        for i,im in enumerate(images):
            a=fig.add_subplot(lower[row,i]);a.imshow(im.reshape(28,28),cmap='gray',vmin=0,vmax=1);a.set_xticks([]);a.set_yticks([])
            for spine in a.spines.values():spine.set_visible(False)
            if i==0:
                a.set_ylabel(['Original','A beta 20','B beta 1','C beta 0.1'][row],fontsize=9,rotation=0,ha='right',va='center')
                a.yaxis.set_label_coords(-.12,.5)
    fig.suptitle('Generation first | reconstruction is a separate task',fontsize=14,y=.99)
    return fig

def save_audit(baseline_labels,intervention_labels,second_labels):
    global AUDIT
    if ready_reason():print('NOT READY: 먼저 변경 실험을 완료하세요.');return
    parsed=[]
    for text in [baseline_labels,intervention_labels,second_labels]:
        labels=[x.strip().lower() for x in text.split(',')]
        if len(labels)!=16 or any(x not in list('0123456789')+['x'] for x in labels):
            AUDIT=None;print('NOT READY: 각 grid의 01–16을 순서대로 0–9 또는 x로 쓰세요. 쉼표로 구분합니다.');return
        parsed.append(labels)
    AUDIT=dict(run_id=VARIANT['id'],second_id=SECOND['id'],baseline=parsed[0],intervention=parsed[1],second=parsed[2])
    print('Visual audit saved. 이 결과는 16개 sample에 대한 주관적 점검이며 전체 생성 성능 점수가 아닙니다.')

def metrics(run):
    out=dict(test_reconstruction_BCE=round(run['rec'],3),test_KL_nats=round(run['kl'],3))
    if AUDIT is not None and VARIANT is not None and SECOND is not None and AUDIT['run_id']==VARIANT['id'] and AUDIT['second_id']==SECOND['id']:
        labels=AUDIT['baseline' if run is BASE else 'intervention' if run is VARIANT else 'second']
        out.update(audited_samples=16,readable_share=round(float(np.mean(np.asarray(labels)!='x')),4),
                   distinct_readable_digits=len(set(labels)-{'x'}),labels=labels)
    return out

def show_comparison():
    reason=ready_reason()
    if reason:print('NOT READY:',reason);return
    fig=comparison_figure();fig.savefig(OUT/'VAE_comparison.png',dpi=140);plt.show();plt.close(fig)
    print('A beta 20',metrics(BASE));print('B beta 1',metrics(VARIANT));print('C beta 0.1',metrics(SECOND))
    print('Grid 01–16을 같은 기준으로 판정하세요. 애매한 이미지는 x입니다. 번호를 바꾸거나 sample을 다시 뽑지 마세요.')

def optional_loss_plot():
    if ready_reason():return
    fig,axes=plt.subplots(1,2,figsize=(9,3))
    for run,label in [(BASE,'A beta 20'),(VARIANT,'B beta 1'),(SECOND,'C beta 0.1')]:
        h=np.array(run['history']);axes[0].plot(h[:,0],h[:,1],label=label);axes[1].plot(h[:,0],h[:,2],label=label)
    for a,title in zip(axes,['Training reconstruction BCE','Training KL (nats per image)']):a.set(xlabel='Epoch',title=title);a.legend()
    fig.tight_layout();plt.show();plt.close(fig)

In [ ]:
#@title MNIST data / MNIST 준비
load_data('auto')
# To check execution without internet: load_data('offline'). Restore auto for submission.

## 1 Observe baseline images / Baseline 관찰
Run the baseline and compare **real MNIST** with **prior/generated** images. Name a visible limitation using sample IDs 01–16. Do not diagnose a learning mechanism from loss alone. / 먼저 실제 이미지와 생성 이미지를 보고 sample 번호로 문제를 짚으세요.

In [ ]:
#@title Baseline training and samples / Baseline 학습과 생성 이미지
run_baseline()
baseline_figure()

## 2 Record the intervention plan / 변경 전 계획
Compare A β=20, B β=1.0 and C β=0.1. Only β changes; model, seed, data, batch order, noise sequence, six epochs, and fixed evaluation `z` stay the same. All runs restart from the same weights. Lower β reduces KL pressure relative to grayscale reconstruction BCE but may weaken the match to the prior; generation must be inspected directly. Predict **both** prior generation and reconstruction before the changes. / A→B와 B→C를 따로 비교하세요.

In [ ]:
#@title Record the plan before training / 변경 전 계획
student_id = '' #@param {type:"string"}
diagnosis = '' #@param {type:"string"}
new_value = 1.0  # prescribed VAE middle run; final run uses beta 0.1
prediction = '' #@param {type:"string"}
second_prediction = '' #@param {type:"string"}
success_rule = '' #@param {type:"string"}
record_plan(student_id, diagnosis, new_value, prediction, success_rule, second_prediction)

## 3 Complete three useful code blocks / 코드 세 칸
These blocks drive the **intervention** training and generated images. The baseline uses the same supplied reference operations, so the comparison changes only the planned hyperparameter when your code passes its self-check. Small checks verify loss, optimizer changes, gradient path, and output format; they do not reveal your experimental answer. / Baseline은 같은 reference 동작으로 실행됩니다. 코드가 막히면 `check_code(1, use_reference=True)`처럼 전환하세요. 감점은 없고, 보고서에는 계산과 역할을 설명해야 합니다. SyntaxError가 나면 답안 줄을 `None`으로 되돌린 다음 전환하세요.

### Code 1 · One VAE optimizer update
Encode x to μ and log variance; sample `z = μ + exp(0.5·logvar)·eps`; compute per-image grayscale BCE and supplied `kl_term(mu, logvar)`; optimize BCE + β·KL. Return two scalar costs. / 실제 update에서 두 항이 어떻게 결합되는지 확인하세요.

In [ ]:
def vae_update(model, opt, x, beta, eps):
    return None  # TODO: replace this block with working code

check_code(1)

### Code 2 · Generate from the prior
Decode supplied prior z, apply sigmoid to grayscale intensities, use eval/no_grad and restore the prior mode. Return NumPy images. / 입력 이미지 없이 z만으로 이미지를 생성합니다.

In [ ]:
def generate_prior(model, z):
    return None  # TODO: replace this block with working code

check_code(2)

### Code 3 · Reconstruct from encoder mean
Encode input x, decode μ without random sampling, apply sigmoid, use eval/no_grad, return NumPy images and restore mode. / 이 결과는 prior generation이 아니라 reconstruction입니다.

In [ ]:
def reconstruct_mean(model, x):
    return None  # TODO: replace this block with working code

check_code(3)

In [ ]:
#@title Run the planned comparison / 계획한 비교 실행
run_variant()

## 4 Compare prior generation and reconstruction / 생성과 reconstruction 비교
Compare the same 64 prior `z` values in A, B and C, with the first 32 displayed. The bottom rows reconstruct the same eight input images. Prior generation has no one-to-one target image. Reconstruction BCE and KL describe other properties and do not directly score generated-image quality. Similar-looking prior samples alone do not establish posterior collapse. Here BCE is a grayscale reconstruction cost; KL is in nats per image. / 생성과 reconstruction을 섞어 결론 내리지 마세요.

In [ ]:
#@title Compare generated images / 생성 이미지 비교
show_comparison()

### Visual audit of 16 generated images / 작은 visual audit
For each generated grid, label images **01–16** with one digit 0–9 if clearly recognizable; otherwise `x`. Use the same rule in every run. `readable share` and `distinct readable digits` summarize only this small subjective sample, not full distribution coverage or classifier accuracy. For GAN, the 64-image panel offers a wider visual check; cite IDs 17–64 if they change your conclusion. No additional labels are required. / 모든 run에 같은 기준을 적용하세요. 애매하면 x입니다. 64장 그리드는 보조 관찰용이며 label 48개를 더 입력할 필요는 없습니다.

In [ ]:
#@title Record image IDs 01–16 / 생성 이미지 판정
baseline_labels = '' #@param {type:"string"}
intervention_labels = '' #@param {type:"string"}
second_labels = '' #@param {type:"string"}
save_audit(baseline_labels, intervention_labels, second_labels)

Optional: inspect training curves only to explain observations. G and D losses have different meanings and do not rank image quality. / Loss curve만으로 생성 성능을 판단하지 마세요.

In [ ]:
#@title Optional curves / 선택 확인
show_curves = False #@param {type:"boolean"}
if show_curves:
    optional_loss_plot()

## 5 Short analysis / 분석 본문
Write **1–2 pages** and then attach the automatic **two-page receipt**. Answer in English or Korean. / 본문 1–2쪽 뒤에 결과 기록 2쪽을 붙여 PDF 하나로 제출하세요.

**Q1 Baseline diagnosis (2 points).** State a visible generation problem, supported by two specific image IDs or features. For VAE, separate prior generation from reconstruction; for GAN, separate unreadable images from repeated readable patterns.

**Q2 Planned changes (2 points).** Explain the two comparisons, your original predictions, what stayed fixed, and one possible cost. Do not rewrite your predictions.

**Q3 What changed (6 points).** For VAE, compare A→B and B→C prior images and all three 16-image audits (readable share and distinct digits). Explain β’s possible effects on reconstruction versus KL and prior generation; distinguish what the samples show from a proposed cause. State one remaining limitation. A negative or mixed result may receive full marks.

**Code and reproducibility (2 points).** Each block: 0.25 for passing execution + 0.25 for a correct explanation of its training/generation role in your report; 0.5 for a READY receipt consistent with the executed notebook. Reference code is accepted without penalty. A numeric typo is deducted once, in the relevant question only.

## 6 Export receipt / 결과 기록 저장
Write an AI-use statement (`none` if unused). The receipt has settings, audit, metrics, image comparison and a fingerprint. It is an appendix; write your own analysis. Before submission run all cells again, review images and receipt, and save outputs. A cache is reused only while settings and code match. / PDF는 `a2_outputs`에 저장됩니다. Colab 왼쪽 Files에서 받으세요. READY는 자료가 갖춰졌다는 뜻이지 분석의 정답 판정이 아닙니다.

In [ ]:
#@title Export the two-page receipt / 결과 기록 PDF
ai_use = '' #@param {type:"string"}
receipt = make_receipt(ai_use)